# 🧱 Part 14: RLHF Alignment: Turning Preferences Into Objectives

> **Previous context**: Training predicts tokens, but useful assistants must optimize for helpful behavior.
> **Goal for this part**: Understand Reward Models, Bradley-Terry preference loss, PPO clipping, KL penalty, and DPO intuition.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. The mismatch

Pretraining teaches continuation. Users want helpful, honest, safe answers. Alignment methods turn preference data into training signals.

## 1. Reward model

A reward model learns to score which response humans prefer. Pairwise comparisons become a supervised learning problem.

## 2. PPO and KL

PPO updates the policy toward higher reward while clipping changes. KL penalty keeps the new model close to the reference model.

## 3. DPO

DPO removes the explicit reward-model step and directly optimizes chosen responses over rejected responses.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] RLHF uses preference data.
- [ ] Reward models score responses.
- [ ] PPO and DPO are two ways to move a model toward preferred behavior.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [ ]:
import numpy as np
import math

np.random.seed(42)

In [ ]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def reward_loss(r_chosen, r_rejected):
    diff = r_chosen - r_rejected
    prob = sigmoid(diff)
    loss = -math.log(max(prob, 1e-10))
    return loss, diff, prob

# Teaching note: follow this line to see the main step.
cases = [
    ('Read the values printed above and connect them to the concept in this cell.', 8.0, 2.0),
    ('Read the values printed above and connect them to the concept in this cell.', 6.0, 5.0),
    ('Read the values printed above and connect them to the concept in this cell.', 3.0, 7.0),
    ('Read the values printed above and connect them to the concept in this cell.', 1.0, 9.0),
]

print('Loss')
print("-" * 70)

for desc, r_c, r_r in cases:
    loss, diff, prob = reward_loss(r_c, r_r)
    print(f"{desc:<25s} {r_c:>6.1f} {r_r:>6.1f} {diff:>8.1f} {prob:>10.4f} {loss:>10.4f}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Loss')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')
print()

def ppo_loss(ratio, advantage, epsilon=0.2):
    'Loss'
    # Teaching note: follow this line to see the main step.
    unclipped = ratio * advantage
    # Teaching note: follow this line to see the main step.
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    # Teaching note: follow this line to see the main step.
    # Teaching note: follow this line to see the main step.
    return -min(unclipped, clipped)


ratios = [0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0]

print('Read the values printed above and connect them to the concept in this cell.')
print('f"{\'ratio\':>8s} {\'unclipped\':>12s} {\'clipped\':>12s} {\'loss\':>10s} {\'Note\'}"')
print("-" * 60)
for r in ratios:
    unclipped = r * 1.0
    clipped = min(r, 1.2) * 1.0
    loss = -min(unclipped, clipped)
    note = ""
    if r > 1.2:
        note = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('f"{\'ratio\':>8s} {\'unclipped\':>12s} {\'clipped\':>12s} {\'loss\':>10s} {\'Note\'}"')
print("-" * 60)
for r in ratios:
    unclipped = r * (-1.0)
    clipped = max(r, 0.8) * (-1.0)
    loss = -min(unclipped, clipped)
    note = ""
    if r < 0.8:
        note = 'Read the values printed above and connect them to the concept in this cell.'
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Training')

In [ ]:
# Teaching note: follow this line to see the main step.
print('Loss')
print()

def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5):
    'Loss'
    # Teaching note: follow this line to see the main step.
    chosen_improvement = logp_chosen - logp_ref_chosen
    # Teaching note: follow this line to see the main step.
    rejected_improvement = logp_rejected - logp_ref_rejected
    
    # Teaching note: follow this line to see the main step.
    diff = beta * (chosen_improvement - rejected_improvement)
    
    # Teaching note: follow this line to see the main step.
    prob = 1 / (1 + math.exp(-diff))
    loss = -math.log(max(prob, 1e-10))
    
    return loss, chosen_improvement, rejected_improvement, diff, prob


scenarios = [
    ('Read the values printed above and connect them to the concept in this cell.',    -1.0, -5.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.', -1.0, -2.5, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',     -5.0, -1.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',            -0.5, -2.0, -3.0, -3.0),
    ('Read the values printed above and connect them to the concept in this cell.',            -4.5, -6.0, -3.0, -3.0),
]

print('Loss')
print("-" * 74)

for desc, lc, lr, ref_c, ref_r in scenarios:
    loss, ci, ri, diff, prob = dpo_loss(lc, lr, ref_c, ref_r)
    print(f"{desc:<28s} {ci:>+10.2f} {ri:>+10.2f} {diff:>10.2f} {loss:>10.4f}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')
print('Read the values printed above and connect them to the concept in this cell.')
print('Loss')